In [232]:
import os
import json
from pathlib import Path
from transformers import AutoModelForTokenClassification, AutoTokenizer
from huggingface_hub import ModelCardData, ModelCard, EvalResult
from torch import nn as nn

from publish_utils import benchmark, get_best_checkpoint, model_stats, load_training_args,\
    make_entity_table, get_first_checkpoint_args, ENTITY_LABELS

MODELS_DIR = Path(os.getcwd()).parent / "models"

In [233]:
current_model = 2

In [ ]:
# Registry: maps each variant to its HF repo and human-readable metadata
MODELS = [
    {
        "dir_name":    "base-bf16-weighted-trainer",
        "repo_id":     "bengid/pii-redaction-deberta-base",
        "model_name":  "DeBERTa-v3-Base PII Redaction",
        "base_model":  "microsoft/deberta-v3-base",
        "size_label":  "Base (86M params)",
        "macro_f1":    0.9557,
        "inference_speed": "~11.7ms on RTX 5070",
        # added after viewing data
        "worst_entities_data": """**Name entities are harder** — The model underperforms on `GIVENNAME` and `LASTNAME` entities:
        
\t| Entity | F1 | Support |
\t|--------|------|---------|
\t| LASTNAME3 | 0.7302 | 105 |
\t| LASTNAME2 | 0.7557 | 313 |
\t| GIVENNAME2 | 0.8175 | 255 |
\t| LASTNAME1 | 0.8497 | 1158 |
\t| GIVENNAME1 | 0.8751 | 904 |

\tLikely causes: performance correlates strongly with training support — 
\tLASTNAME1/GIVENNAME1 (primary occurrences, ~900–1100 examples) score 
\tsignificantly higher than LASTNAME2/3 (secondary/tertiary occurrences, 
\t105–313 examples). Additionally, names are inherently context-dependent: 
\twithout surrounding cues like titles or formal structure, the model has 
\tless signal to distinguish them from non-PII tokens — even the 
\tbest-supported name entities (LASTNAME1, GIVENNAME1) fall notably below 
\tthe macro F1 of 0.9557, suggesting names are a structurally harder 
\tcategory regardless of support.
        """,
        "best_for": "Accuracy",
        "recommendation": """
**Recommended when** accuracy is the priority and compute is not a 
constraint. Best overall performance (macro F1: 0.9557), particularly 
on rare entity types. Well-suited for offline batch processing, 
compliance pipelines, or any server-side deployment where an extra 
~5ms of latency is acceptable. Inference: ~11.7ms on RTX 5070."""
    },
    {
        "dir_name":    "small-bf16-weighted-trainer",
        "repo_id":     "bengid/pii-redaction-deberta-small",
        "model_name":  "DeBERTa-v3-Small PII Redaction",
        "base_model":  "microsoft/deberta-v3-small",
        "size_label":  "Small (44M params)",
        "macro_f1":    0.9517,
        "inference_speed": "~6.5ms on RTX 5070",
        # added after viewing data
        "worst_entities_data": """**Name entities are harder** — The model underperforms on `GIVENNAME` and `LASTNAME` entities:

\t| Entity | F1 | Support |
\t|--------|------|---------|
\t| LASTNAME2 | 0.7570 | 313 |
\t| LASTNAME3 | 0.7822 | 105 |
\t| GIVENNAME2 | 0.8102 | 255 |
\t| LASTNAME1 | 0.8501 | 1158 |
\t| GIVENNAME1 | 0.8640 | 904 |

\tLikely causes: performance correlates strongly with training support — 
\tLASTNAME1/GIVENNAME1 (primary occurrences, ~900–1100 examples) score 
\tsignificantly higher than LASTNAME2/3 (secondary/tertiary occurrences, 
\t105–313 examples). Additionally, names are inherently context-dependent: 
\twithout surrounding cues like titles or formal structure, the model has 
\tless signal to distinguish them from non-PII tokens — even the 
\tbest-supported name entities (LASTNAME1, GIVENNAME1) fall notably below 
\tthe macro F1 of 0.9557, suggesting names are a structurally harder 
\tcategory regardless of support.""",
        "best_for": "Latency",
        "recommendation": """
**Recommended when** you need the best latency with minimal accuracy 
tradeoff. Despite being 44M parameters, its 6-layer architecture 
makes it the fastest of the three (~6.5ms on RTX 5070) — nearly 2x faster 
than base while retaining strong performance. 
Best fit for real-time APIs or high-throughput services."""
    },
    {
        "dir_name":    "xsmall-bf16-weighted-trainer",
        "repo_id":     "bengid/pii-redaction-deberta-xsmall",
        "model_name":  "DeBERTa-v3-XSmall PII Redaction",
        "base_model":  "microsoft/deberta-v3-xsmall",
        "size_label":  "XSmall (22M params)",
        "macro_f1":    0.9424,
        "inference_speed": "~11.6ms on RTX 5070 [1]",
    # added after viewing data
        "worst_entities_data": """**Name entities are harder** — The model underperforms on `GIVENNAME` and `LASTNAME` entities:           

\t| Entity | F1 | Support |
\t|--------|------|---------|
\t| LASTNAME2 | 0.7279 | 313 |
\t| LASTNAME3 | 0.7423 | 105 |
\t| GIVENNAME2 | 0.7675 | 255 |
\t| LASTNAME1 | 0.8087 | 1158 |
\t| GIVENNAME1 | 0.8294 | 904 |

\tLikely causes: performance correlates strongly with training support — 
\tLASTNAME1/GIVENNAME1 (primary occurrences, ~900–1100 examples) score 
\tsignificantly higher than LASTNAME2/3 (secondary/tertiary occurrences, 
\t105–313 examples). Additionally, names are inherently context-dependent: 
\twithout surrounding cues like titles or formal structure, the model has 
\tless signal to distinguish them from non-PII tokens — even the 
\tbest-supported name entities (LASTNAME1, GIVENNAME1) fall notably below 
\tthe macro F1 of 0.9557, suggesting names are a structurally harder 
\tcategory regardless of support.""",
        "best_for": "Memory",
        "recommendation": """
**Recommended when** memory footprint is the hard constraint — 
edge deployments, CPU inference, or environments where the 22M 
parameter count matters more than raw latency. 
#### Latency Note
Note that despite being the smallest model, RTX 5070 latency (~11.6ms) 
is comparable to base due to its identical 12-layer depth; sequential 
layer passes dominate GPU latency more than hidden dimension width. The 
advantage over base is memory, not speed."""
    },
]

In [235]:
# for model_cfg in MODELS:
#     model_dir = MODELS_DIR / model_cfg["dir_name"]
    
#     # Get best checkpoint from trainer state
#     best_ckpt, _, _ = get_best_checkpoint(model_dir)
    
#     print(f"Pushing {model_cfg['model_name']} from {best_ckpt}")
    
#     model = AutoModelForTokenClassification.from_pretrained(best_ckpt)
#     tokenizer = AutoTokenizer.from_pretrained(best_ckpt)
    
#     model.push_to_hub(model_cfg["repo_id"], commit_message="Best checkpoint")
#     tokenizer.push_to_hub(model_cfg["repo_id"])
    
#     print(f"✓ {model_cfg['repo_id']}")

In [236]:
print(len(ENTITY_LABELS))

27


In [237]:
# text = "John Smith's account number is 4111-1111-1111-1111 and his email is john@example.com"

# import torch
# import gc

# for model in MODELS:
#     ms = benchmark(model["repo_id"], text, n_warmup=20, n_runs=100)
#     print(f"{model['size_label']}: {ms:.1f}ms")
    
#     # Clear between models
#     torch.cuda.empty_cache()
#     gc.collect()

In [238]:
def get_poor_f1s(log: dict, n: int=10):
    f1s = []
    for key, value in log.items():
        if "f1" in key:
            f1s.append((key, value))
    poor_f1s = sorted(f1s, key=lambda x: x[1])[:n]
    return poor_f1s

In [239]:
# for info in MODELS:
#     model_dir = MODELS_DIR / info["dir_name"]
#     print(get_best_checkpoint(model_dir))

In [240]:
current_info = MODELS[current_model]
current_model_dir = MODELS_DIR/current_info["dir_name"]

In [241]:
best_ckpt, state, best_log = get_best_checkpoint(current_model_dir)
training_args = load_training_args(best_ckpt)

f1  = best_log.get("eval_f1", 0)
pre = best_log.get("eval_precision", 0)
rec = best_log.get("eval_recall", 0)
acc = best_log.get("eval_accuracy", 0)
entity_table = make_entity_table(best_log)

base_kwargs = dict(
    task_type="token_classification",
    dataset_name="ai4privacy/pii-masking-300k",
    dataset_type="ai4privacy/pii-masking-300k",
    dataset_split="validation",
)

eval_results = [
    EvalResult(**{**base_kwargs, "metric_type": "f1",        "metric_value": f1}),
    EvalResult(**{**base_kwargs, "metric_type": "precision", "metric_value": pre}),
    EvalResult(**{**base_kwargs, "metric_type": "recall",    "metric_value": rec}),
]

In [242]:
print(make_entity_table(best_log, worst=6))

| Entity | F1 | Support |
|--------|------|---------|
| LASTNAME2 | 0.7279 | 313 |
| LASTNAME3 | 0.7423 | 105 |
| GIVENNAME2 | 0.7675 | 255 |
| LASTNAME1 | 0.8087 | 1158 |
| GIVENNAME1 | 0.8294 | 904 |
| DATE | 0.9233 | 837 |



In [243]:
f"""
- **Name entities are harder** — The model underperforms on `GIVENNAME` and `LASTNAME` entities:
\n{make_entity_table(best_log, worst=5)}
Likely causes: performance correlates strongly with training support — 
LASTNAME1/GIVENNAME1 (primary occurrences, ~900–1100 examples) score 
significantly higher than LASTNAME2/3 (secondary/tertiary occurrences, 
105–313 examples). Additionally, names are inherently context-dependent: 
without surrounding cues like titles or formal structure, the model has 
less signal to distinguish them from non-PII tokens — even the 
best-supported name entities (LASTNAME1, GIVENNAME1) fall notably below 
the macro F1 of {f1}, suggesting names are a structurally harder 
category regardless of support.
"""
print(current_info["worst_entities_data"])

**Name entities are harder** — The model underperforms on `GIVENNAME` and `LASTNAME` entities:           

	| Entity | F1 | Support |
	|--------|------|---------|
	| LASTNAME2 | 0.7279 | 313 |
	| LASTNAME3 | 0.7423 | 105 |
	| GIVENNAME2 | 0.7675 | 255 |
	| LASTNAME1 | 0.8087 | 1158 |
	| GIVENNAME1 | 0.8294 | 904 |

	Likely causes: performance correlates strongly with training support — 
	LASTNAME1/GIVENNAME1 (primary occurrences, ~900–1100 examples) score 
	significantly higher than LASTNAME2/3 (secondary/tertiary occurrences, 
	105–313 examples). Additionally, names are inherently context-dependent: 
	without surrounding cues like titles or formal structure, the model has 
	less signal to distinguish them from non-PII tokens — even the 
	best-supported name entities (LASTNAME1, GIVENNAME1) fall notably below 
	the macro F1 of 0.9557, suggesting names are a structurally harder 
	category regardless of support.


In [244]:
card_data = ModelCardData(
    language="en",
    license="other",
    library_name="transformers",
    base_model=current_info["base_model"],
    tags=["token-classification", "pii", "ner", "deberta", "privacy", "named-entity-recognition"],
    datasets=["ai4privacy/pii-masking-300k"],
    metrics=["f1", "precision", "recall"],
    model_name=current_info["model_name"],
    pipeline_tag="token-classification",
    eval_results=eval_results,
)

In [245]:
description = (
    f"Fine-tuned [{current_info['base_model']}](https://huggingface.co/{current_info['base_model']}) for "
    f"Named Entity Recognition targeting 27 PII entity types. "
    f"Trained on the English subset of [ai4privacy/pii-masking-300k](https://huggingface.co/datasets/ai4privacy/pii-masking-300k) "
    f"with a class-weighted `CrossEntropyLoss`. Achieves **{f1:.4f} macro-F1** on the validation set."
)

In [246]:
stage1_args = get_first_checkpoint_args(current_model_dir)
stage2_args = training_args  # loaded from best checkpoint

def get_precision(args) -> str:
    if args.bf16: return "bf16"
    if args.fp16: return "fp16"
    return "fp32"

# Stage 1
stage1_lr             = stage1_args.learning_rate
stage1_bs             = stage1_args.per_device_train_batch_size
stage1_grad_acc       = stage1_args.gradient_accumulation_steps
stage1_effective_bs   = stage1_bs * stage1_grad_acc
stage1_precision      = get_precision(stage1_args)
stage1_scheduler      = stage1_args.lr_scheduler_type.value
stage1_warmup         = stage1_args.warmup_steps

# Stage 2
stage2_lr             = stage2_args.learning_rate
stage2_bs             = stage2_args.per_device_train_batch_size
stage2_grad_acc       = stage2_args.gradient_accumulation_steps
stage2_effective_bs   = stage2_bs * stage2_grad_acc
stage2_precision      = get_precision(stage2_args)
stage2_scheduler      = stage2_args.lr_scheduler_type.value
stage2_warmup         = stage2_args.warmup_steps



In [247]:
# Build the model comparison table for the model card
comparison_rows = []
for m in MODELS:
    row = (
        f"| [{m['model_name']}](https://huggingface.co/{m['repo_id']}) "
        f"| {m['macro_f1']:.4f} "
        f"| {m['size_label']} "
        f"| {m['inference_speed']} "
        f"| {m['best_for']} |"
    )
    comparison_rows.append(row)
    
if current_info['model_name'] == "DeBERTa-v3-XSmall PII Redaction":
    note = "[1] see [Latency Note](#latency-note) for latency explanation"
else:
    note = "[1] see [x-small card](https://huggingface.co/bengid/pii-redaction-deberta-xsmall/#latency-note) for latency explanation"

comparison_rows.append("\n" + note)
comparison_table = "\n".join(comparison_rows)
print(comparison_table)


| [DeBERTa-v3-Base PII Redaction](https://huggingface.co/bengid/pii-redaction-deberta-base) | 0.9557 | Base (86M params) | ~11.7ms on RTX 5070 | Accuracy |
| [DeBERTa-v3-Small PII Redaction](https://huggingface.co/bengid/pii-redaction-deberta-small) | 0.9517 | Small (44M params) | ~6.5ms on RTX 5070 | Latency |
| [DeBERTa-v3-XSmall PII Redaction](https://huggingface.co/bengid/pii-redaction-deberta-xsmall) | 0.9424 | XSmall (22M params) | ~11.6ms on RTX 5070 [1] | Memory |

[1] see [Latency Note](#latency-note) for latency explanation


In [248]:
from transformers import pipeline

text = "My name is John Smith and my email is john@example.com"
for m in MODELS:
    pipe = pipeline(
        task="token-classification",
        model=m["repo_id"],
        aggregation_strategy="simple",
    )
    entities = pipe(text)
    print(entities)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[]


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'entity_group': 'EMAIL', 'score': np.float32(0.60965174), 'word': 'john', 'start': 37, 'end': 42}]


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[]


In [249]:
text = "She lives at 742 Evergreen Terrace, Springfield, IL 62704."

for model_cfg in MODELS:
    model_dir = MODELS_DIR / model_cfg["dir_name"]
    best_ckpt, _, _ = get_best_checkpoint(model_dir)
    
    
    model= AutoModelForTokenClassification.from_pretrained(best_ckpt)
    tokenizer = AutoTokenizer.from_pretrained(best_ckpt)
    
    pipe = pipeline(
        task="token-classification",
        model=model_cfg["repo_id"],
        aggregation_strategy="first",
        device=0  # omit for CPU
    )
    model_cfg["example_output"] = pipe(text)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
content = f"""---
{card_data.to_yaml()}
---

# {current_info["model_name"]}

{description}

{current_info["recommendation"]}

## Usage

```python
from transformers import pipeline

pipe = pipeline(
    "token-classification",
    model="{current_info["repo_id"]}",
    aggregation_strategy="first",
    device=0  # omit for CPU
)

text = "She lives at 742 Evergreen Terrace, Springfield, IL 62704."
entities = pipe(text)
print(entities)
```

## Training Data

Filtered subset of [ai4privacy/pii-masking-300k](https://huggingface.co/datasets/ai4privacy/pii-masking-300k),
restricted to **English-language examples only** (`language == "en"`).  
The full dataset is multilingual; this model targets English text only.

| Split | Full Dataset | English Subset |
|-------|-------------|----------------|
| Train | 177,677 | 29,908 |
| Validation | 47,728 | 3,973 |
| Test | — | 3,973 |

**Preprocessing:**
- Dropped `CARDISSUER` entity class (little support)
- Validation set split 50/50 into validation and test

## Training Procedure

Two-phase Fine-tuning (frozen backbone → unfrozen) from [{current_info['base_model']}](https://huggingface.co/{current_info['base_model']}) 
using a weighted token-classification trainer and stage-specific learning rates.

### Hyperparameters
| Parameter | Stage 1 (frozen backbone) | Stage 2 (full fine-tune) |
|-----------|--------------------------|--------------------------|
| Learning rate         | {stage1_lr}           | {stage2_lr}          |
| LR scheduler          | {stage1_scheduler}    | {stage2_scheduler}   |
| Warmup steps          | {stage1_warmup}       | {stage2_warmup}      |
| Batch size (per device) | {stage1_bs}         | {stage2_bs}          |
| Gradient accumulation | {stage1_grad_acc}     | {stage2_grad_acc}    |
| Effective batch size  | {stage1_effective_bs} | {stage2_effective_bs}|
| Precision             | {stage1_precision}    | {stage2_precision}   |
| Weight decay          | {stage1_args.weight_decay} | {stage2_args.weight_decay} |
| Seed                  | {stage1_args.seed}    | {stage2_args.seed}   |

## Evaluation

Evaluated on the English validation subset (3,973 examples) at the best checkpoint.

| Metric | Value |
|--------|-------|
| **F1 (macro)** | **{f1:.4f}** |
| Precision | {pre:.4f} |
| Recall | {rec:.4f} |
| Token Accuracy | {acc:.4f} |

### Per-Entity F1

{entity_table}

## Limitations

- **English only** — trained exclusively on English text; performance on other languages is undefined.
- **Max 512 tokens** — inherited from DeBERTa's positional embeddings. Longer documents should be chunked.
- **Name entities are harder** — The model underperforms on `GIVENNAME` and `LASTNAME` entities:

	Likely causes: performance correlates strongly with training support — 
	LASTNAME1/GIVENNAME1 (primary occurrences, ~900-1100 examples) score 
	significantly higher than LASTNAME2/3 (secondary/tertiary occurrences, 
	105-313 examples). Additionally, names are inherently context-dependent: 
	without surrounding cues like titles or formal structure, the model has 
	less signal to distinguish them from non-PII tokens — even the 
	best-supported name entities (LASTNAME1, GIVENNAME1) fall notably below 
	the macro F1 of {f1:.4f}, suggesting names are a structurally harder 
	category regardless of support.
- **Not a redaction tool by itself** — this model detects and labels PII spans; downstream redaction/masking logic must be implemented separately.
- **Subword labeling convention** — following the HuggingFace token classification convention, only the first subword of each word was assigned its NER label during training; continuation subwords were assigned `-100` (ignored by the loss). The practical consequence is that the model predicts `O` with high confidence on continuation subwords, which can cause partial detection of multi-subword entities (e.g. `john@example.com` returned as only `john`) when using `aggregation_strategy="simple"`. Use `aggregation_strategy="first"` for inference, which is consistent with this training convention.

## Intended Use

**Intended uses:**
- Detecting and labeling PII spans in English text for downstream redaction or pseudonymization pipelines.
- Privacy compliance tooling (GDPR, CCPA, HIPAA).
- Pre-processing step before storing or sharing user-generated content.

**Out-of-scope uses:**
- Non-English text.
- Real-time high-stakes medical or legal decision-making without human review.
- As a sole compliance mechanism — model errors are expected; human auditing is recommended.

## Model Comparison

| Model | Macro F1 | Params (non-embedding) | Inference Speed | Best For |
|-------|----------|------------------------|-----------------|----------|
{comparison_table}

## License

The model weights are released for research and non-commercial use,
consistent with the training data license
([ai4privacy/pii-masking-300k](https://huggingface.co/datasets/ai4privacy/pii-masking-300k)).
Users should review the dataset license before commercial deployment.
""" + """
## Citation

If you use this model, please cite the base model architecture and the training dataset:

**Base model (DeBERTa-v3):**
```bibtex
@misc{he2021debertav3,
      title={DeBERTaV3: Improving DeBERTa using ELECTRA-Style Pre-Training with Gradient-Disentangled Embedding Sharing}, 
      author={Pengcheng He and Jianfeng Gao and Weizhu Chen},
      year={2021},
      eprint={2111.09543},
      archivePrefix={arXiv},
      primaryClass={cs.CL}
}
```

**Training dataset:**
```bibtex
@misc{ai4privacy2023pii,
  title     = {PII Masking 300k},
  author    = {Ai4Privacy},
  year      = {2023},
  publisher = {Hugging Face},
  doi       = {10.57967/hf/1995},
  url       = {https://huggingface.co/datasets/ai4privacy/pii-masking-300k}
}
```
"""

In [251]:
card_dirs = Path("../model_cards")
card_dirs.mkdir(exist_ok=True)
with open(card_dirs/(current_info["dir_name"]+"-readme.md"), "w") as f:
    f.write(content)

In [252]:

card = ModelCard(content=content)
card.push_to_hub(current_info["repo_id"])

CommitInfo(commit_url='https://huggingface.co/bengid/pii-redaction-deberta-xsmall/commit/8efc9fc7eb439988a9af97447435c1883c03f88f', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='8efc9fc7eb439988a9af97447435c1883c03f88f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/bengid/pii-redaction-deberta-xsmall', endpoint='https://huggingface.co', repo_type='model', repo_id='bengid/pii-redaction-deberta-xsmall'), pr_revision=None, pr_num=None)